# 📊 Análisis de Datos — Respuesta a Preguntas de Negocio
**Materia:** Herramientas de Software para Big Data  
**Fuente:** Tablas Hive — esquema `inumet`

**Preguntas a responder:**
1. ¿Qué departamentos concentran las temperaturas extremas y en qué meses ocurren?
2. ¿Existe correlación entre los días de menor insolación solar y los de mayor precipitación?
3. ¿Cuál es la evolución mensual de temperatura en Montevideo vs el interior?
4. ¿Qué estaciones registran los vientos más intensos y en qué estación del año?
5. ¿Cómo varía la presión entre estaciones costeras e interiores y qué relación tiene con las precipitaciones?

**Preguntas con visualización en Jupyter:** 1, 3, 4  
**Preguntas que van a /anl y SuperSet:** 2, 5

## 1. Inicializacion de Spark con soporte Hive

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "pandas", "matplotlib", "seaborn"])

### Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['figure.figsize'] = (13, 5)
matplotlib.rcParams['font.size'] = 11

### Sesion Spark

In [ ]:
spark = SparkSession.builder \
    .appName("INUMET_Preguntas") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.sql("USE inumet")

ANL = "hdfs://localhost:9000/anl/obligatorio"
MESES_LABELS = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
ORDEN_MESES = ['Enero','Febrero','Marzo','Abril','Mayo','Jun','Julio',
               'Agosto','Septiembre','Octubre','Noviembre','Diciembre']

print(f"Spark {spark.version} listo")
print("\nTablas disponibles en inumet:")
spark.sql("SHOW TABLES").show(truncate=False)

### Funciones auxiliares

In [ ]:
def agregar_fecha_mes(df):
    return df.withColumn(
        "fecha_mes",
        F.concat(
            F.col("anio").cast("string"),
            F.lit("-"),
            F.lpad(F.col("mes").cast("string"), 2, "0"),
            F.lit("-01")
        )
    )


def guardar_anl(df, nombre_tabla, carpeta):
    path = f"{ANL}/{carpeta}"
    df.write.mode("overwrite").parquet(path)
    print(f"Resultado guardado en {path}")

    spark.sql(f"DROP TABLE IF EXISTS inumet.{nombre_tabla}")
    df.write \
        .mode("overwrite") \
        .option("path", path) \
        .saveAsTable(f"inumet.{nombre_tabla}")

    print(f"Tabla Hive inumet.{nombre_tabla} creada")
    spark.sql(f"SELECT * FROM inumet.{nombre_tabla}").show(5, truncate=False)


def pivot_mensual(pdf, valor):
    pivot = pdf.pivot(index='departamento', columns='mes', values=valor)
    pivot = pivot.reindex(columns=range(1, 13))
    pivot.columns = MESES_LABELS
    return pivot

---
## Pregunta 1
### ¿Qué departamentos concentran las temperaturas extremas (máximas y mínimas) y en qué meses ocurren?

### Consulta

In [ ]:
df_p1 = spark.sql("""
    SELECT
        e.departamento,
        t.nombre_mes,
        t.mes,
        ROUND(MAX(f.temp_aire), 2) AS temp_maxima,
        ROUND(MIN(f.temp_aire), 2) AS temp_minima,
        ROUND(AVG(f.temp_aire), 2) AS temp_promedio
    FROM inumet.fact_temperatura f
    JOIN inumet.dim_estaciones e ON f.estacion_id = e.estacion_id
    JOIN inumet.dim_tiempo     t ON f.fecha = t.fecha
    GROUP BY e.departamento, t.nombre_mes, t.mes
    ORDER BY e.departamento, t.mes
""").cache()

df_p1.show(20, truncate=False)

### Visualizacion 1 — Temperaturas extremas por departamento

In [ ]:
# Temperatura maxima absoluta por departamento
df_max = df_p1.groupBy("departamento") \
    .agg(
        F.max("temp_maxima").alias("temp_maxima"),
        F.min("temp_minima").alias("temp_minima")
    ) \
    .orderBy(F.desc("temp_maxima")) \
    .toPandas()

fig, ax = plt.subplots()
x = range(len(df_max))

bars_max = ax.bar(x, df_max['temp_maxima'], label='Maxima', color='tomato', alpha=0.85, width=0.4, align='edge')
bars_min = ax.bar([i - 0.4 for i in x], df_max['temp_minima'], label='Minima', color='steelblue', alpha=0.85, width=0.4, align='edge')

# Etiquetas maximas — arriba de la barra en rojo
for bar, val in zip(bars_max, df_max['temp_maxima']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val}°', ha='center', va='bottom', fontsize=9, color='tomato', fontweight='bold')

# Etiquetas minimas — debajo del eje en azul
for bar, val in zip(bars_min, df_max['temp_minima']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.3,
            f'{val}°', ha='center', va='top', fontsize=9, color='steelblue', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df_max['departamento'], rotation=30, ha='right')
ax.set_ylabel('Temperatura (°C)')
ax.set_title('Pregunta 1 — Temperaturas extremas por departamento')
ax.legend(loc='upper left')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

### Visualizaciones 2, 3 y 4 — Heatmaps mensuales

In [ ]:
pdf_p1 = df_p1.toPandas()

heatmaps_p1 = [
    ('temp_promedio', 'Pregunta 1 — Temperatura promedio mensual por departamento (°C)'),
    ('temp_minima',   'Pregunta 1 — Temperatura minima mensual por departamento (°C)'),
    ('temp_maxima',   'Pregunta 1 — Temperatura maxima mensual por departamento (°C)'),
]

for columna, titulo in heatmaps_p1:
    pivot = pivot_mensual(pdf_p1, columna)

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlBu_r',
                linewidths=0.5, ax=ax, cbar_kws={'label': '°C'})
    ax.set_title(titulo)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Departamento')
    plt.tight_layout()
    plt.show()

**Conclusión Pregunta 1**

Los departamentos del norte del país concentran las temperaturas más extremas. Salto lidera con la máxima absoluta más alta (41.9°C), seguido por Tacuarembó (41.4°C) y Soriano (41.3°C); todos en el mes de Enero. Esto responde a que el norte y centro del país están más alejados de la influencia termorreguladora del Río de la Plata y el Atlántico.

Las temperaturas máximas extremas se concentran en los meses de **verano (diciembre, enero y febrero)**, con enero y febrero siendo los meses más calurosos en todos los departamentos. Las mínimas más bajas ocurren en **julio**, siendo Soriano el departamento con el registro más frío (-6.3°C en julio) y Artigas el segundo (-4.1°C aproximadamente).

Los departamentos costeros como Colonia y Rocha muestran rangos térmicos más moderados: sus máximas son menores y sus mínimas son más altas que en el interior, efecto típico de la regulación térmica que ejerce el agua.

---
## Pregunta 2 — Va a /anl y SuperSet
### ¿Existe correlación entre los días de menor insolación solar y los de mayor precipitación?

### Consulta

In [ ]:
df_p2 = spark.sql("""
    SELECT
        t.anio,
        t.mes,
        t.nombre_mes,
        ROUND(AVG(i.horas_insolacion), 2) AS insolacion_promedio,
        ROUND(SUM(p.precip_horario), 2)   AS precipitacion_total
    FROM inumet.fact_insolacion i
    JOIN inumet.fact_precipitacion p ON i.fecha = p.fecha AND i.estacion_id = p.estacion_id
    JOIN inumet.dim_tiempo t ON i.fecha = t.fecha
    GROUP BY t.anio, t.mes, t.nombre_mes
    ORDER BY t.anio, t.mes
""")

df_p2 = agregar_fecha_mes(df_p2)
df_p2.show(20, truncate=False)

### Guardado en /anl y tabla Hive

In [ ]:
# Guardar resultado en /anl para SuperSet
guardar_anl(
    df_p2,
    nombre_tabla="anl_pregunta2",
    carpeta="pregunta2_insolacion_precipitacion"
)

---
## Pregunta 3
### ¿Cuál es la evolución mensual de temperatura en Montevideo comparada con el Interior?

### Consulta

In [ ]:
# QUERY — Montevideo vs resto de departamentos
df_p3 = spark.sql("""
    SELECT
        CASE WHEN e.departamento = 'Montevideo' THEN 'Montevideo' ELSE 'Interior' END AS region,
        t.anio,
        t.mes,
        t.nombre_mes,
        ROUND(AVG(f.temp_aire), 2) AS temp_promedio
    FROM inumet.fact_temperatura f
    JOIN inumet.dim_estaciones e ON f.estacion_id = e.estacion_id
    JOIN inumet.dim_tiempo     t ON f.fecha = t.fecha
    GROUP BY
        CASE WHEN e.departamento = 'Montevideo' THEN 'Montevideo' ELSE 'Interior' END,
        t.anio, t.mes, t.nombre_mes
    ORDER BY region, t.anio, t.mes
""")

df_p3.show(20, truncate=False)

### Preparacion para graficos

In [ ]:
pdf_p3 = df_p3.toPandas()
pdf_p3['periodo'] = pdf_p3['anio'].astype(str) + '-' + pdf_p3['mes'].astype(str).str.zfill(2)

mvd = pdf_p3[pdf_p3['region'] == 'Montevideo']
resto = pdf_p3[pdf_p3['region'] == 'Interior']

### Visualizacion 1 — Linea temporal Montevideo vs Resto del pais

In [ ]:
fig, ax = plt.subplots()
ax.plot(mvd['periodo'], mvd['temp_promedio'], marker='o', label='Montevideo', color='steelblue', linewidth=2, markersize=3)
ax.plot(resto['periodo'], resto['temp_promedio'], marker='o', label='Interior', color='tomato', linewidth=2, markersize=3)
ax.set_xlabel('Periodo')
ax.set_ylabel('Temperatura promedio (°C)')
ax.set_title('Pregunta 3 — Evolucion mensual de temperatura: Montevideo vs Interior')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

### Visualizacion 2 — Diferencia Montevideo menos Resto del pais por periodo

In [ ]:
df_diff = pdf_p3.groupby(['periodo', 'region'])['temp_promedio'].mean().unstack()
df_diff['diferencia'] = df_diff['Montevideo'] - df_diff['Interior']

fig, ax = plt.subplots()
colores = ['steelblue' if v >= 0 else 'tomato' for v in df_diff['diferencia']]
ax.bar(df_diff.index, df_diff['diferencia'], color=colores, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Periodo')
ax.set_ylabel('Diferencia (°C)')
ax.set_title('Pregunta 3 — Diferencia de temperatura Montevideo menos Interior')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Conclusión Pregunta 3**

La evolución mensual muestra que el interior del país tiene un comportamiento térmico más extremo que Montevideo a lo largo del año. En verano, el interior supera consistentemente a Montevideo en temperatura promedio, mientras que en invierno desciende más por debajo.

Montevideo, al estar sobre el Río de la Plata, actúa como regulador térmico: los veranos son más frescos y los inviernos más templados que en el resto del país. Esta diferencia es más pronunciada en los meses de julio y agosto, donde el interior puede estar 3 a 5°C por debajo de Montevideo, y en enero-febrero donde el interior puede superar a Montevideo en 2 a 4°C.

Este patrón se repite consistentemente en todos los años del dataset (2020-2026), confirmando que es una característica estructural del clima uruguayo y no una anomalía puntual.

---
## Pregunta 4
### ¿Qué estaciones registran los vientos más intensos y en qué estación del año se concentran?

### Consulta

In [ ]:
df_p4 = spark.sql("""
    SELECT
        f.estacion_id,
        e.departamento,
        t.estacion_anio,
        ROUND(AVG(f.int_viento), 2) AS viento_promedio,
        ROUND(MAX(f.int_viento), 2) AS rafaga_maxima
    FROM inumet.fact_viento f
    JOIN inumet.dim_estaciones e ON f.estacion_id = e.estacion_id
    JOIN inumet.dim_tiempo     t ON f.fecha = t.fecha
    GROUP BY f.estacion_id, e.departamento, t.estacion_anio
    ORDER BY viento_promedio DESC
""").cache()

df_p4.show(20, truncate=False)

### Preparacion para graficos

In [ ]:
pdf_p4 = df_p4.toPandas()

### Visualizacion 1 — Viento promedio por estacion y temporada

In [ ]:
pivot_viento = pdf_p4.pivot(index='estacion_id', columns='estacion_anio', values='viento_promedio')

fig, ax = plt.subplots(figsize=(13, 5))
pivot_viento.plot(kind='bar', ax=ax, colormap='Set2', alpha=0.85)
ax.set_xlabel('Estacion meteorologica')
ax.set_ylabel('Viento promedio (km/h)')
ax.set_title('Pregunta 4 — Intensidad de viento por estacion y temporada del año')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Temporada')
plt.tight_layout()
plt.show()

### Visualizacion 2 — Rafaga maxima por estacion

In [ ]:
df_max_viento = spark.sql("""
    SELECT
        f.estacion_id,
        e.departamento,
        ROUND(AVG(f.int_viento), 2) AS viento_promedio,
        ROUND(MAX(f.int_viento), 2) AS rafaga_maxima
    FROM inumet.fact_viento f
    JOIN inumet.dim_estaciones e ON f.estacion_id = e.estacion_id
    GROUP BY f.estacion_id, e.departamento
    ORDER BY rafaga_maxima DESC
""").toPandas()

fig, ax = plt.subplots()

bars_raf = ax.barh(df_max_viento['estacion_id'], df_max_viento['rafaga_maxima'], label='Rafaga maxima', color='tomato', alpha=0.8)
bars_prom = ax.barh(df_max_viento['estacion_id'], df_max_viento['viento_promedio'], label='Viento promedio', color='mediumseagreen', alpha=0.8)

# Etiquetas rafaga maxima — al final de la barra
for bar, val in zip(bars_raf, df_max_viento['rafaga_maxima']):
    ax.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val} km/h', ha='right', va='center', fontsize=9, color='white', fontweight='bold')

# Etiquetas viento promedio — dentro de la barra
for bar, val in zip(bars_prom, df_max_viento['viento_promedio']):
    ax.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val} km/h', ha='right', va='center', fontsize=9, color='white', fontweight='bold')

ax.set_xlabel('Velocidad (km/h)')
ax.set_title('Pregunta 4 — Rafaga maxima y viento promedio por estacion')
ax.legend()
plt.tight_layout()
plt.show()

### Visualizacion extra P4 — Viento promedio por temporada del año

In [ ]:
df_temporada = spark.sql("""
    SELECT
        t.estacion_anio,
        ROUND(AVG(f.int_viento), 2) AS viento_promedio
    FROM inumet.fact_viento f
    JOIN inumet.dim_tiempo t ON f.fecha = t.fecha
    GROUP BY t.estacion_anio
    ORDER BY viento_promedio DESC
""").toPandas()

orden = ['verano', 'otonio', 'invierno', 'primavera']
df_temporada['estacion_anio'] = pd.Categorical(df_temporada['estacion_anio'], categories=orden, ordered=True)
df_temporada = df_temporada.sort_values('estacion_anio')

fig, ax = plt.subplots()
colores = ['tomato', 'sandybrown', 'steelblue', 'mediumseagreen']
bars = ax.bar(df_temporada['estacion_anio'], df_temporada['viento_promedio'], color=colores, alpha=0.85)

for bar, val in zip(bars, df_temporada['viento_promedio']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val} km/h', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Temporada del año')
ax.set_ylabel('Viento promedio (km/h)')
ax.set_title('Pregunta 4 — Viento promedio por temporada del año (todas las estaciones)')
ax.set_ylim(0, df_temporada['viento_promedio'].max() + 1)
plt.tight_layout()
plt.show()

### Viento promedio mensual por estacion — norte vs sur

In [ ]:
df_norte_sur = spark.sql("""
    SELECT
        e.estacion_id,
        e.departamento,
        t.mes,
        t.nombre_mes,
        ROUND(AVG(f.int_viento), 2) AS viento_promedio
    FROM inumet.fact_viento f
    JOIN inumet.dim_estaciones e ON f.estacion_id = e.estacion_id
    JOIN inumet.dim_tiempo     t ON f.fecha = t.fecha
    WHERE e.departamento IN ('Salto', 'Artigas', 'Colonia', 'Rocha', 'Montevideo')
    GROUP BY e.estacion_id, e.departamento, t.mes, t.nombre_mes
    ORDER BY e.departamento, t.mes
""").toPandas()

df_norte_sur['nombre_mes'] = pd.Categorical(df_norte_sur['nombre_mes'], categories=ORDEN_MESES, ordered=True)

pivot_ns = df_norte_sur.pivot_table(index='departamento', columns='mes', values='viento_promedio')
pivot_ns = pivot_ns.reindex(columns=range(1, 13))
pivot_ns.columns = MESES_LABELS

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_ns, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'km/h'})
ax.set_title('Viento promedio mensual por departamento — Norte vs Sur')
ax.set_xlabel('Mes')
ax.set_ylabel('Departamento')

# Marcar semana de turismo (abril) y primavera (oct)
ax.add_patch(plt.Rectangle((3, 0), 1, len(pivot_ns), fill=False,
                           edgecolor='blue', lw=3, label='Semana de Turismo (Abr)'))
ax.add_patch(plt.Rectangle((9, 0), 1, len(pivot_ns), fill=False,
                           edgecolor='green', lw=3, label='Primavera (Oct)'))
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1))

plt.tight_layout()
plt.show()

**Conclusión Pregunta 4**

El Aeropuerto de Melilla (Montevideo) es la estación con mayor viento promedio sostenido (8.99 km/h en verano), seguida por Colonia G3 y Paso de los Toros. Esto refleja la influencia de los vientos del sudeste que afectan principalmente la zona costera sur del país.

Sin embargo, la estación con las **ráfagas máximas más intensas** es Artigas G3, con un pico de 36.0 km/h registrado en primavera. Esto es consistente con el ingreso de masas de aire cálido del norte que en primavera generan frentes severos en el norte del país.

La **primavera (setiembre-noviembre)** es la estación del año con vientos más fuertes en casi todas las estaciones, seguida por el verano. El invierno y el otoño registran los vientos más bajos en promedio. Este patrón responde al mayor contraste térmico entre masas de aire en la transición primaveral.

**Análisis adicional — Hipótesis sobre vientos regionales**

Durante el análisis de la Pregunta 4 se planteó la hipótesis de que el norte del país
(Salto, Artigas) experimentaría vientos más intensos en abril (Semana de Turismo) 
mientras que el sur lo haría en primavera (octubre), lo que explicaría culturalmente 
la tradición de remontar cometas en distintas épocas según la región.

Sin embargo, los datos meteorológicos de INUMET no respaldan esta hipótesis. 
En abril, Salto registra apenas 5.3 km/h y Artigas 7.3 km/h, valores que se 
encuentran entre los más bajos del año para ambas estaciones. El pico de vientos 
en el norte ocurre en septiembre-octubre, coincidiendo con el resto del país.

Un hallazgo inesperado es que **Montevideo es la estación meteorológica más 
ventosa del país de forma consistente**, con valores superiores a 7 km/h durante 
todo el año y un pico de 9.3 km/h en noviembre.

Esta discrepancia entre la percepción cultural y los datos medidos puede explicarse 
por factores que las estaciones de INUMET no capturan, como vientos locales a nivel 
de calle, canalización del viento por la geografía urbana, o simplemente que la 
tradición cultural responde a otros factores además de la velocidad promedio del viento.

---
## Pregunta 5 — Va a /anl y SuperSet
### ¿Cómo varía la presión entre estaciones costeras e interiores y qué relación tiene con las precipitaciones?

### Consulta

In [ ]:
df_p5 = spark.sql("""
    SELECT
        e.zona,
        t.anio,
        t.mes,
        t.nombre_mes,
        ROUND(AVG(pr.pres_atm_mar), 2) AS presion_promedio,
        ROUND(SUM(p.precip_horario), 2) AS precipitacion_total
    FROM inumet.fact_presion pr
    JOIN inumet.fact_precipitacion p ON pr.fecha = p.fecha AND pr.estacion_id = p.estacion_id
    JOIN inumet.dim_estaciones e ON pr.estacion_id = e.estacion_id
    JOIN inumet.dim_tiempo     t ON pr.fecha = t.fecha
    GROUP BY e.zona, t.anio, t.mes, t.nombre_mes
    ORDER BY e.zona, t.anio, t.mes
""")

df_p5 = agregar_fecha_mes(df_p5)
df_p5.show(20, truncate=False)

### Guardado en /anl y tabla Hive

In [ ]:
# Guardar resultado en /anl para SuperSet
guardar_anl(
    df_p5,
    nombre_tabla="anl_pregunta5",
    carpeta="pregunta5_presion_precipitacion"
)

## Resumen de resultados

In [ ]:
print("Tablas finales en Hive:")
spark.sql("SHOW TABLES IN inumet").show(truncate=False)

In [ ]:
spark.stop()
print("Sesion Spark cerrada.")